# Company Research & Brochure Assistant

Give it a company name and website URL. It scrapes the site, picks out the pages worth reading (About, Careers, etc.), writes a short brochure about the company, and then lets you ask follow-up questions about it — with real conversation memory, using GPT, Grok, or Groq's free tier.

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import tiktoken

In [ ]:
# Initialize and constants

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
groq_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("OpenAI API key looks good so far")
else:
    print("There might be a problem with your OpenAI API key? Please visit the troubleshooting notebook!")

openai = OpenAI(api_key=api_key)
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

MODEL = "gpt-4.1-mini"             
LINK_MODEL = "gpt-5-nano"          

encoding = tiktoken.get_encoding("cl100k_base")

In [ ]:
# Get all links from the OpenAI website

links = fetch_website_links("https://openai.com/")
links

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
Decide which links are most relevant to include in a brochure about the company,
such as an About page, Company page, or Careers/Jobs page.

IMPORTANT RULES:
- Only include links on the company's OWN domain. Ignore external sites such as
  LinkedIn, Twitter/X, GitHub, Discord, status pages, and external job boards.
- Do not include Terms of Service, Privacy, or email (mailto:) links.
- Return at most 5 links.
- Convert relative links such as "/about" into the full https URL.

Respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://openai.com/"))

In [ ]:
def select_relevant_links(url):
    try:
        response = openai.chat.completions.create(
            model=LINK_MODEL,
            messages=[
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": get_links_user_prompt(url)}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Link selection failed: {e}")
        return {"links": []}

In [ ]:
select_relevant_links("https://openai.com/")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links['links']:
        try:
            page_content = fetch_website_contents(link["url"])
            result += f"\n\n### Link: {link['type']}\n{page_content}"
        except Exception as e:
            print(f"Could not fetch {link['url']}: {e}")

    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://openai.com/"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages
from a company website and creates a short brochure about the company.

The brochure is intended for:
- prospective customers
- investors
- potential employees

Include information such as:
- Company overview
- Products and services
- What makes the company different
- Customers or users
- Company culture
- Careers and job opportunities

Only use information provided in the website content.
Do not make up facts.

Respond in Markdown.
"""

In [ ]:
MAX_TOKENS = 3_000  # keeps the prompt from getting too big or too expensive

def get_brochure_user_prompt(company_name, url):
    header = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    content = fetch_page_and_all_relevant_links(url)
    tokens = encoding.encode(content)[:MAX_TOKENS]
    print(f"Using {len(tokens)} tokens of content")
    return header + encoding.decode(tokens)

In [ ]:
get_brochure_user_prompt("OpenAI", "https://openai.com")

In [ ]:
def create_brochure(company_name, url, client=openai, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result

In [ ]:
#create_brochure("OpenAI", "https://openai.com")


create_brochure("OpenAI", "https://openai.com", groq, "openai/gpt-oss-120b")

In [ ]:
def stream_brochure(company_name, url, client=openai, model=MODEL):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
brochure_text = stream_brochure("OpenAI", "https://openai.com")

## New: chat about the company

Ask follow-up questions and get answers that remember what you already asked.

In [ ]:
def chat_about_company(brochure_text, client=openai, model=MODEL):
    messages = [
        {"role": "system", "content": f"Answer questions about the company based only on this brochure:\n\n{brochure_text}"}
    ]
    print("Ask anything. Type 'quit' to stop.\n")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit"):
            break
        messages.append({"role": "user", "content": user_input})

        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
        reply = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            reply += chunk.choices[0].delta.content or ""
            update_display(Markdown(reply), display_id=display_handle.display_id)

        messages.append({"role": "assistant", "content": reply})

    return messages

In [ ]:
conversation = chat_about_company(brochure_text)